In [1]:
# PHASE 2 - CCSM (Cross-Centre Style Mixing) for lunit : all seeds 42/43/44 in ONE notebook (re-running resumes; finished arms are skipped).
# Kaggle: GPU on, Internet on.  ~25-30 min on a T4
# OUTPUT: /kaggle/working/checkpoints/ccsm_lunit_<arm>_s<seed>_best.pt  -> upload all 9 files (3 seeds x 3 arms) as a NEW dataset
#         version of peesarisathvikreddy/adjclv2-checkpoints, then run phase2_D6_lunit_eval.ipynb.
!pip install -q datasets timm transformers

"""
CCSM = MixStyle (feature-statistics mixing after the FIRST block/stage, partners drawn from a DIFFERENT centre) inside SimCLR pretraining,
continued from Lunit ViT-S/16.   Arms:  control (SimCLR, no mixing) | ms (CCSM, p=0.5, alpha=0.1) | fms (CCSM + Fourier amplitude swap, window 0.05).
Pool: UNLABELED patches from centres {0,1,3,4}; centre 2 (primary test centre) is never used.  Patches are stored at 96px and bilinearly
up-sampled to 224px on the GPU (exactly what the evaluation does).  Backbone LR = 0.1 x head LR (as in the adjacency runs).
"""
import os, time, random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')

FAMILY = 'lunit'
SPECS = {   # kind, hub, input size, patches per centre, batch (pairs), epochs
    'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, per_center=5000, batch=64, epochs=6),
    'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, per_center=5000, batch=64, epochs=6),
    'kaiko': dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, per_center=5000, batch=64, epochs=6),
    'lunit': dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, per_center=5000, batch=64, epochs=6),
    'phikon': dict(kind='hf', hub='owkin/phikon', size=224, per_center=3000, batch=64, epochs=6),
}
SP = SPECS[FAMILY]

class Cfg:
    HF = "wltjr1007/Camelyon17-WILDS"
    SEEDS = [42, 43, 44]
    POOL_SEED = 42
    PRE_CENTERS = [0, 1, 3, 4]
    N_PER_CENTER = SP['per_center']
    EPOCHS = SP['epochs']; BATCH = SP['batch']; LR = 3e-4; BB_MULT = 0.1; WD = 0.05; TAU = 0.1
    FOURIER_BETA = 0.05; LAM_MAX = 1.0
    MS_P = 0.5; MS_ALPHA = 0.1
    ARMS = ['control', 'ms', 'fms']
    LOAD_SIZE = 96
    SIZE = SP['size']
    CKPT_DIR = "/kaggle/working/checkpoints"
    PRETRAINED = True

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_pool(hf):
    from tqdm.auto import tqdm
    centers = np.array(hf['center'])
    rng = np.random.RandomState(Cfg.POOL_SEED); idx = []
    for c in Cfg.PRE_CENTERS:
        idx += list(rng.choice(np.where(centers == c)[0], Cfg.N_PER_CENTER, replace=False))
    imgs = np.zeros((len(idx), Cfg.LOAD_SIZE, Cfg.LOAD_SIZE, 3), np.uint8)
    for k, i in enumerate(tqdm(idx, desc='loading pool')):
        im = hf[int(i)]['image'].convert('RGB')
        if im.size != (Cfg.LOAD_SIZE, Cfg.LOAD_SIZE): im = im.resize((Cfg.LOAD_SIZE, Cfg.LOAD_SIZE))
        imgs[k] = np.asarray(im)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.tensor(centers[idx])

def base_aug(x):
    B = x.size(0)
    a = torch.empty(B, device=x.device).uniform_(0.4, 1.0); s = a.sqrt()
    tx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s); ty = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s)
    fx = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0); fy = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    th = torch.zeros(B, 2, 3, device=x.device); th[:, 0, 0] = s * fx; th[:, 1, 1] = s * fy; th[:, 0, 2] = tx; th[:, 1, 2] = ty
    x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode='reflection', align_corners=False)
    r = lambda lo, hi: torch.empty(B, 1, 1, 1, device=x.device).uniform_(lo, hi)
    x = x * r(0.8, 1.2)
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * r(0.8, 1.2) + m
    g = x.mean(1, keepdim=True); x = (x - g) * r(0.8, 1.2) + g
    return x.clamp(0, 1)

def partner_index(c):
    mask = (c[:, None] != c[None, :]).float()
    mask[mask.sum(1) == 0] = 1.0
    return torch.multinomial(mask, 1).squeeze(1)

def fourier_swap(x, j, beta, lam_max):
    B, _, H, W = x.shape
    Fx = torch.fft.fft2(x.float()); amp = torch.fft.fftshift(Fx.abs(), dim=(-2, -1)); ph = Fx.angle()
    h = int(beta * H); c = H // 2; sl = slice(c - h, c + h + 1)
    lam = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, lam_max)
    new = amp.clone()
    new[..., sl, sl] = (1 - lam) * amp[..., sl, sl] + lam * amp[j][..., sl, sl]
    new = torch.fft.ifftshift(new, dim=(-2, -1))
    return torch.fft.ifft2(new * torch.exp(1j * ph)).real.clamp(0, 1)

class PhikonBackbone(nn.Module):
    """owkin/phikon -> CLS token (identical to Part I and to the adjacency E notebooks)."""
    def __init__(self, hub, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.vit = AutoModel.from_pretrained(hub) if pretrained else AutoModel.from_config(AutoConfig.from_pretrained(hub))
        self.dim = self.vit.config.hidden_size
    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]

def make_backbone():
    if SP['kind'] == 'hf':
        bb = PhikonBackbone(SP['hub'], Cfg.PRETRAINED)
        bb.vit.gradient_checkpointing_enable()
        first = bb.vit.encoder.layer[0] if hasattr(bb.vit, 'encoder') else bb.vit.layers[0]      # transformers 4.x / 5.x
        return bb, bb.dim, first
    import timm
    bb = timm.create_model(SP['hub'], pretrained=Cfg.PRETRAINED, num_classes=0)
    dim = bb.num_features
    target = bb.blocks[0] if hasattr(bb, 'blocks') else bb.layers[0]
    return bb, dim, target

class Net(nn.Module):
    """MixStyle is applied by a forward hook on the first block/stage. The mixing decision (on/off, lambda, partner) is drawn ONCE per
    step outside the forward pass, so it is identical under gradient-checkpoint recomputation."""
    def __init__(self, mixstyle=False):
        super().__init__()
        self.backbone, dim, target = make_backbone()
        self.head = nn.Sequential(nn.Linear(dim, 512), nn.ReLU(), nn.Linear(512, 128))
        self.mixstyle = mixstyle; self.mix = None          # (lam [B,1,..], partner idx) or None
        if mixstyle:
            target.register_forward_hook(self._hook)
    def _hook(self, mod, inp, out):
        if not (self.training and self.mix is not None):
            return out
        h = out[0] if isinstance(out, tuple) else out
        dims = (1,) if h.dim() == 3 else (1, 2)            # tokens [B,N,C] or NHWC [B,H,W,C]
        lam, j = self.mix
        lam = lam.view(-1, *([1] * (h.dim() - 1))).to(h.dtype)
        mu = h.mean(dims, keepdim=True); sd = (h.var(dims, keepdim=True) + 1e-6).sqrt()
        h2 = (h - mu) / sd * (lam * sd + (1 - lam) * sd[j]) + (lam * mu + (1 - lam) * mu[j])
        return (h2,) + tuple(out[1:]) if isinstance(out, tuple) else h2
    def forward(self, x):
        return F.normalize(self.head(self.backbone(x)), dim=1)

def nt_xent(z, tau):
    n = z.size(0) // 2
    sim = z @ z.t() / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, tgt), (sim.argmax(1) == tgt).float().mean().item()

def prep(x):
    if x.shape[-1] != Cfg.SIZE:
        x = F.interpolate(x, size=(Cfg.SIZE, Cfg.SIZE), mode='bilinear', align_corners=False)
    return (x - MEAN.to(x.device)) / STD.to(x.device)

def pretrain(arm, imgs, cen, seed):
    set_seed(seed)
    ms = arm in ('ms', 'fms'); use_f = arm == 'fms'
    net = Net(mixstyle=ms).to(dev).train()
    opt = torch.optim.AdamW([{'params': net.backbone.parameters(), 'lr': Cfg.LR * Cfg.BB_MULT},
                             {'params': net.head.parameters(), 'lr': Cfg.LR}], weight_decay=Cfg.WD)
    spe = len(imgs) // Cfg.BATCH
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, [Cfg.LR * Cfg.BB_MULT, Cfg.LR], total_steps=Cfg.EPOCHS * spe, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=dev.type == 'cuda'); chance = float(np.log(2 * Cfg.BATCH - 1))
    for ep in range(Cfg.EPOCHS):
        perm = torch.randperm(len(imgs)); L, A, t0 = 0.0, 0.0, time.time()
        for b in range(spe):
            ix = perm[b * Cfg.BATCH:(b + 1) * Cfg.BATCH]
            x = imgs[ix].to(dev).float() / 255; c = cen[ix].to(dev)
            v1, v2 = base_aug(x), base_aug(x)
            if use_f:
                v2 = fourier_swap(v2, partner_index(c), Cfg.FOURIER_BETA, Cfg.LAM_MAX)
            if ms:
                if random.random() < Cfg.MS_P:
                    cc = torch.cat([c, c]); lam = torch.distributions.Beta(Cfg.MS_ALPHA, Cfg.MS_ALPHA).sample((cc.numel(),)).to(dev)
                    net.mix = (lam, partner_index(cc))
                else:
                    net.mix = None
            with torch.autocast(dev.type, enabled=dev.type == 'cuda'):
                z = net(prep(torch.cat([v1, v2])))
            loss, acc = nt_xent(z.float(), Cfg.TAU)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            L += loss.item(); A += acc
        print(f"  [{arm} s{seed}] ep {ep + 1}/{Cfg.EPOCHS} loss {L / spe:.3f} (chance {chance:.2f}) inst-acc {A / spe:.3f} | {(time.time() - t0) / 60:.1f} min")
    if L / spe > 0.95 * chance:
        print(f"  !! [{arm}] loss still at chance -> do NOT use this arm")
    net.mix = None
    return net.eval(), L / spe

def main(hf=None, tensors=None):
    imgs, cen = tensors if tensors is not None else load_pool(hf)
    os.makedirs(Cfg.CKPT_DIR, exist_ok=True)
    for seed in Cfg.SEEDS:
        for arm in Cfg.ARMS:
            path = f"{Cfg.CKPT_DIR}/ccsm_{FAMILY}_{arm}_s{seed}_best.pt"
            if os.path.exists(path):
                print(f"  [{arm} s{seed}] already done -> skip"); continue
            t = time.time(); net, L = pretrain(arm, imgs, cen, seed)
            torch.save({'model_state_dict': net.state_dict(), 'arm': arm, 'seed': seed, 'epoch': Cfg.EPOCHS, 'best_val': L}, path)
            print(f"  [{arm} s{seed}] saved {path} | {(time.time() - t) / 60:.1f} min")
    print("\nDONE. Files:", sorted(os.listdir(Cfg.CKPT_DIR)))

if __name__ == '__main__':
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Cfg.HF); parts = [ds[k] for k in ds.keys()]
    hf = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    main(hf)


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

loading pool:   0%|          | 0/20000 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.7MB            

model.safetensors: downloading bytes:           |  0.00B            

  [control s42] ep 1/6 loss 0.258 (chance 4.84) inst-acc 0.993 | 1.9 min
  [control s42] ep 2/6 loss 0.076 (chance 4.84) inst-acc 0.999 | 2.1 min
  [control s42] ep 3/6 loss 0.057 (chance 4.84) inst-acc 1.000 | 2.2 min
  [control s42] ep 4/6 loss 0.049 (chance 4.84) inst-acc 1.000 | 2.2 min
  [control s42] ep 5/6 loss 0.044 (chance 4.84) inst-acc 1.000 | 2.2 min
  [control s42] ep 6/6 loss 0.040 (chance 4.84) inst-acc 1.000 | 2.2 min
  [control s42] saved /kaggle/working/checkpoints/ccsm_lunit_control_s42_best.pt | 13.0 min
  [ms s42] ep 1/6 loss 0.329 (chance 4.84) inst-acc 0.980 | 2.2 min
  [ms s42] ep 2/6 loss 0.101 (chance 4.84) inst-acc 0.998 | 2.2 min
  [ms s42] ep 3/6 loss 0.071 (chance 4.84) inst-acc 0.999 | 2.2 min
  [ms s42] ep 4/6 loss 0.060 (chance 4.84) inst-acc 0.999 | 2.2 min
  [ms s42] ep 5/6 loss 0.052 (chance 4.84) inst-acc 0.999 | 2.2 min
  [ms s42] ep 6/6 loss 0.049 (chance 4.84) inst-acc 0.999 | 2.2 min
  [ms s42] saved /kaggle/working/checkpoints/ccsm_lunit_ms_s42